## SILMA TTS V2 - Streaming API Call Example

In [ ]:
import requests
import numpy as np
import time
import IPython.display as ipd

In [ ]:
# Create a persistent session
session = requests.Session()

In [ ]:


def stream_waveform(text, voice="sarah"):


    url = "https://2orycinwjnbwqz-7860.proxy.runpod.net/stream"

    payload = {"text": text, "voice_id": voice, "cfg": 0.5, "creativity": 0.5}

    start_time = time.perf_counter()
    carry_over = b""
    first_byte_received = False

    try:
        with session.post(url, json=payload, stream=True) as r:

            if r.status_code == 200:

                # Iterate over raw bytes as they arrive from the network
                for chunk in r.iter_content(chunk_size=None):

                    if chunk:
                        if not first_byte_received:

                            ttft = (time.perf_counter() - start_time) * 1000
                            print(f"⏱️ TTFT: {ttft:.2f} ms")
                            first_byte_received = True

                        # Combine carry-over from previous chunk with new data
                        current_data = carry_over + chunk

                        # Calculate how many full 4-byte floats we have
                        num_floats = len(current_data) // 4
                        cut_off = num_floats * 4

                        # Separate valid bytes from the new remainder
                        valid_bytes = current_data[:cut_off]
                        carry_over = current_data[cut_off:]

                        if valid_bytes:
                            # Convert to waveform and yield back to the caller
                            waveform = np.frombuffer(valid_bytes, dtype=np.float32)

                            yield waveform

            ##status code != 200
            else:

                print(f"Server rejected the request with status: {r.status_code}")

                try:

                    error_data = r.json()
                    print(f"Error message: {error_data.get('detail')}")

                except requests.exceptions.JSONDecodeError:
                    print(f"Error message: '{r.text}'")

                yield None

    except Exception as e:
        print(f"Streaming Error: {e}")
        yield None

## max input length is 250 chars
text = """
ايش من الأغاني القديمة الرائعة الي تحسونها من وقت سمعتوها وحتى اليوم مازالت حتى هالوقت تلامس قلوبكم؟ او من الكتب او الأفلام او حتى الموسيقى الشعبية.
""".strip()

all_audio_cunks = []
sample_rate = 24000

print("Starting Stream...")
for audio_chunk in stream_waveform(text):
    if audio_chunk is not None:
        print(f"Received waveform chunk: {len(audio_chunk)} samples")
        all_audio_cunks.append(audio_chunk)

if len(all_audio_cunks)>0:
    full_waveform = np.concatenate(all_audio_cunks)

    ## Play full streamed audio
    ipd.display(ipd.Audio(full_waveform, rate=sample_rate, autoplay=True))


Starting Stream...
⏱️ TTFT: 1575.50 ms
Received waveform chunk: 2208 samples
Received waveform chunk: 2047 samples
Received waveform chunk: 3937 samples
Received waveform chunk: 1022 samples
Received waveform chunk: 3262 samples
Received waveform chunk: 4930 samples
Received waveform chunk: 7170 samples
Received waveform chunk: 1022 samples
Received waveform chunk: 7170 samples
Received waveform chunk: 9214 samples
Received waveform chunk: 4096 samples
Received waveform chunk: 3074 samples
Received waveform chunk: 1022 samples
Received waveform chunk: 7728 samples
Received waveform chunk: 16384 samples
Received waveform chunk: 4096 samples
Received waveform chunk: 3074 samples
Received waveform chunk: 1022 samples
Received waveform chunk: 4096 samples
Received waveform chunk: 3074 samples
Received waveform chunk: 1022 samples
Received waveform chunk: 7170 samples
Received waveform chunk: 13310 samples
Received waveform chunk: 3992 samples
Received waveform chunk: 8192 samples
Received 